# IE Tower VPR — test the trained model

**Goal.** Given a photo of an indoor location somewhere in the IE Tower (above-ground floor or basement level), retrieve the most similar gallery frames and predict which floor the photo was taken on.

## Before running this notebook

Make sure the pipeline has been built once on your machine:

```bash
pip install -r requirements.txt
python scripts/sync_drive_data.py             # pull all team videos
python scripts/extract_frames_and_update_csv.py
python scripts/assign_splits.py
python scripts/run_pipeline.py --model-name dinov2_vits14_hires   # best config
python scripts/run_evaluation.py
```

After that, the four files this notebook needs are on disk:

* `outputs/index/gallery.index` — FAISS Flat-IP index over the gallery embeddings.
* `outputs/embeddings/gallery_metadata.csv` — one row per indexed frame (image_path, label, split, …).
* `outputs/embeddings/gallery_info.json` — side-car file recording which backbone produced the embeddings (used here to load the matching feature extractor automatically).
* `outputs/results/evaluation.json` — Top-K accuracy and mAP for the held-out queries.

## What this notebook does

1. Loads the FAISS index, gallery metadata and the matching feature extractor.
2. Reports the saved evaluation metrics so you can see how well the model performs before testing on your own photo.
3. Picks a random held-out query frame and shows the prediction with Top-K matches (sanity check that the pipeline is wired up correctly).
4. Lets you point at any image on disk to test the model on a fresh photo.

## How the model works

* **Feature extractor.** By default DINOv2 ViT-S/14 at its native 518×518 resolution (Meta AI, self-supervised on 142 M images), frozen. This is a drop-in upgrade from the original ResNet50 baseline — see `outputs/_compare/comparison.json` for the bake-off results.
* **Gallery.** Every frame whose `split == "gallery"` in `data/metadata/dataset.csv`, encoded with the feature extractor and stored in a FAISS Flat-IP index. L2-normalised embeddings are searched via cosine similarity.
* **Prediction.** Top-K cosine search, then majority vote over the floor labels of those K matches. Confidence is the size of the winning vote (e.g. `4/5` means 4 of the 5 nearest neighbours agree).

Every backbone described in the README is supported automatically — the notebook reads `gallery_info.json` to pick the matching feature extractor for whatever index is currently on disk.

In [ ]:
# Cell 1 — Imports and project-path setup
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Walk up from notebooks/ to the repo root so `src` is importable.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.features.extract_embeddings import extract_pil_image_embedding
from src.features.models import get_feature_extractor, resolve_device
from src.features.transforms import get_image_transform
from src.retrieval.faiss_utils import load_index
from src.retrieval.search import search_index
from src.utils.config import EMBEDDINGS_DIR, INDEX_DIR, RESULTS_DIR

print(f"Repo root:   {REPO_ROOT}")
print(f"Embeddings:  {EMBEDDINGS_DIR}")
print(f"Index:       {INDEX_DIR}")
print(f"Eval JSON:   {RESULTS_DIR}")

In [ ]:
# Cell 2 — Load the saved gallery + the matching feature extractor.
INDEX_PATH = INDEX_DIR / "gallery.index"
METADATA_PATH = EMBEDDINGS_DIR / "gallery_metadata.csv"
INFO_PATH = EMBEDDINGS_DIR / "gallery_info.json"
EVAL_PATH = RESULTS_DIR / "evaluation.json"

for path in (INDEX_PATH, METADATA_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing artifact: {path}.\n"
            "Run `python scripts/run_pipeline.py` and `python scripts/run_evaluation.py` first."
        )

info = json.loads(INFO_PATH.read_text(encoding="utf-8")) if INFO_PATH.exists() else {}
model_name = info.get("model_name", "resnet50")
print(f"Backbone in use:  {model_name}")
if info:
    print(f"Embedding dim:    {info.get('embedding_dim')}")
    print(f"Metric:           {info.get('metric', 'cosine')}")
    print(f"Gallery rows:     {info.get('num_rows')}")

device = resolve_device(None)
model, embedding_dim = get_feature_extractor(model_name)
transform = get_image_transform(model_name)

index = load_index(INDEX_PATH)
metadata = pd.read_csv(METADATA_PATH)

print()
print(f"Device:           {device}")
print(f"Floors covered:   {sorted(metadata['label'].unique())}")
print(f"Gallery rows:     {len(metadata)}")

In [ ]:
# Cell 3 — Show the evaluation metrics saved by run_evaluation.py.
if EVAL_PATH.exists():
    eval_results = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
    print("Headline metrics:")
    print(f"  Top-1 accuracy: {eval_results.get('top_1_accuracy', 0):.1%}")
    print(f"  Top-5 accuracy: {eval_results.get('top_5_accuracy', 0):.1%}")
    print(f"  mAP:            {eval_results.get('mAP', 0):.1%}")
    print(f"  Held-out queries: {eval_results.get('num_queries', 'unknown')}")

    per_class = eval_results.get("per_class") or {}
    if per_class:
        print()
        print("Per-class Top-1 accuracy (sorted by floor):")
        def _key(label):
            text = str(label)
            if text.startswith("basement"):
                try:
                    return (0, int(text.replace("basement", "")))
                except ValueError:
                    return (0, 0)
            if text.startswith("floor"):
                try:
                    return (1, int(text.replace("floor", "")))
                except ValueError:
                    return (1, 0)
            return (2, 0)
        for label in sorted(per_class.keys(), key=_key):
            acc = per_class[label]["top1_accuracy"]
            n = per_class[label]["queries"]
            bar = "#" * int(round(acc * 30))
            print(f"  {label:12s}  {acc:6.1%}  ({n:3d} queries)  {bar}")
else:
    print("No evaluation.json yet. Run `python scripts/run_evaluation.py` and re-run this cell.")

In [ ]:
# Cell 4 — Helper functions for running and visualising a query.
def predict_floor(image_path, top_k=5, metric="cosine"):
    """Run a single retrieval query.

    Returns ``(predicted_label, votes, results)`` where ``results`` is a list
    of ``SearchResult`` objects with ``rank``, ``score``, ``label`` and
    ``image_path`` attributes (see ``src/retrieval/search.py``).
    """
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)

    image = ImageOps.exif_transpose(Image.open(image_path)).convert("RGB")

    embedding = extract_pil_image_embedding(
        image=image,
        model=model,
        transform=transform,
        device=device,
        normalize=metric == "cosine",
    )
    results = search_index(
        query_embeddings=embedding,
        index=index,
        metadata=metadata,
        top_k=top_k,
        metric=metric,
    )[0]

    if not results:
        return None, 0, []
    counts = Counter(r.label for r in results if r.label)
    top_label, top_count = counts.most_common(1)[0]
    return top_label, top_count, results


def show_results(query_path, top_k=5, ground_truth=None):
    """Run a query and render the prediction + Top-K thumbnails inline.

    Pass ``ground_truth="floor10"`` etc. to colour-code the result.
    """
    predicted, votes, results = predict_floor(query_path, top_k=top_k)
    confidence = votes / max(len(results), 1)

    header = f"Predicted floor: {predicted}  (confidence {votes}/{len(results)} = {confidence:.0%})"
    if ground_truth is not None:
        verdict = "\u2713 correct" if predicted == ground_truth else f"\u2717 wrong (true: {ground_truth})"
        header = f"{header}  \u2014  {verdict}"
    print(header)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(3 * (len(results) + 1), 3))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])

    query_img = ImageOps.exif_transpose(Image.open(query_path)).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title("Query")
    axes[0].axis("off")

    for ax, result in zip(axes[1:], results):
        try:
            img = Image.open(result.image_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "missing", ha="center", va="center")
        title = f"#{result.rank} {result.label}\nscore {result.score:.3f}"
        if ground_truth is not None:
            title += "  \u2713" if result.label == ground_truth else "  \u2717"
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return predicted, votes, results

## Sanity check: random held-out query

We pick a random row whose `split == "query"` and ask the model to retrieve the closest gallery matches. Because every query frame was held out of the gallery (`scripts/assign_splits.py`), this is a real retrieval and not a self-match.

In [ ]:
# Cell 5 — Pick a random held-out query and run the model on it.
csv_path = REPO_ROOT / "data" / "metadata" / "dataset.csv"
all_rows = pd.read_csv(csv_path)
query_rows = all_rows[all_rows["split"] == "query"]

if query_rows.empty:
    raise RuntimeError("No query rows in dataset.csv. Did you run assign_splits.py?")

sample = query_rows.sample(1, random_state=42).iloc[0]
sample_path = REPO_ROOT / sample["image_path"]
if not sample_path.exists():
    sample_path = REPO_ROOT / "data" / sample["image_path"]

print(f"Ground-truth floor: {sample['label']}")
print(f"Query image:        {sample_path}")
_ = show_results(sample_path, top_k=5, ground_truth=str(sample["label"]))

## Test with your own image

Drop any photo of an IE Tower indoor location somewhere on disk and put the path in the cell below. JPG and PNG both work. Phone photos are handled correctly — EXIF orientation is baked into the pixels before the model sees the image, so portrait shots are not classified upside down.

In [ ]:
# Cell 6 — Edit the path below and re-run this cell to test your own image.
your_image = REPO_ROOT / "data" / "processed_frames" / "floor3" / "f03_hallway_main_frame_000010.jpg"

# Replace the line above with your own absolute or relative path, e.g.:
# your_image = Path(r"C:\Users\Professor\Desktop\my_photo.jpg")
# your_image = Path("/Users/professor/Desktop/my_photo.jpg")

_ = show_results(your_image, top_k=5)

## How to read the output

* **Predicted floor** — the majority vote across the Top-K retrieved gallery frames.
* **Confidence** — how many of the Top-K matched the predicted label. `5/5 = 100%` means the model is very sure; `2/5 = 40%` means it is hesitant and the retrieval space looks ambiguous around your photo.
* **Score** — inner product (cosine similarity for L2-normalised embeddings). Higher is closer. On this dataset, anything above ~0.85 is a very strong visual match.
* **Ground truth** (only shown for the random sanity-check sample) — the true label according to the dataset CSV. The notebook colour-codes whether each Top-K match is correct (`✓`) or not (`✗`).

## Known failure modes

Documented in detail in the README under "Analysis: why isn't accuracy higher?". The two most common ones to look out for when the prediction is wrong:

1. **Vertical confusion.** The IE Tower hallways and elevators repeat the same layout from one above-ground floor to the next. Frozen ResNet50 / DINOv2 cannot read floor-number signage, so they often confuse `floorN_hallway_left` with `floor(N±k)_hallway_left`. The per-class breakdown printed by Cell 3 makes this visible: basements (architecturally distinct) hit ~80 % Top-1, while middle above-ground floors are closer to 30–50 % Top-1.
2. **Lighting / device shift.** All gallery frames came from a single phone walkthrough per area at one moment in time. Photos taken under very different lighting (bright daylight vs evening) or with a different camera tend to drop in confidence. The headline fix here is to capture a second pass per floor under different conditions, not to change the model.

If your prediction is wrong, inspect the Top-K thumbnails first: if they all look visually similar to your query but on the *wrong* floor, you are looking at the vertical-confusion mode. The next iteration of this project plans to address it with triplet-loss fine-tuning and an OCR head for floor-number signage.